<div style="background:#10263d;padding:28px 32px;border-radius:14px;color:#f8fafc;">
<p style="color:#62d5c8;font-size:12px;letter-spacing:2px;margin:0;">TRAFFIC INTELLIGENCE · PEMS08</p>
<h1 style="color:#ffffff;margin:10px 0;">Spatio-Temporal Graph Neural Network</h1>
<p style="font-size:17px;color:#d9e6f2;margin:0;">Prediksi traffic flow dengan GCN + GRU</p>
<p style="color:#aac0d4;margin-top:16px;">Data → pemeriksaan → split kronologis → graf → training → evaluasi → inference</p>
</div>

**Tujuan:** memakai riwayat flow, occupancy, dan speed dari beberapa sensor untuk memprediksi traffic flow berikutnya.

| Komponen | Konfigurasi awal |
|---|---|
| Input | 12 langkah historis × seluruh sensor × 3 fitur |
| Target | Flow, fitur indeks 0 |
| Horizon | 12 langkah; ubah `output_steps=1` untuk menyamai notebook awal |
| Split | 70% train · 10% validation · 20% test, berurutan |
| Model | Dua lapis GCN residual + GRU + koreksi terhadap nilai terakhir |
| Pembanding | Persistence: ulangi pengamatan terakhir untuk seluruh horizon |

> **Status hasil:** notebook ini belum dilatih ulang pada PEMS08 karena berkas `pems08.npz` tidak disertakan. Tidak ada angka akurasi baru yang diklaim. Mode reference opsional hanya untuk memeriksa alur dengan data referensi.


## Cara menjalankan

1. Di Colab, aktifkan GPU jika tersedia. CPU juga didukung.
2. Periksa `data_path`, urutan fitur, serta `sample_minutes` pada konfigurasi. Path Google Drive awal dipertahankan.
3. Jalankan semua sel dari atas ke bawah. Jika hanya ingin mencoba alurnya, set `use_reference_data=True`.
4. Lihat tabel evaluasi dan grafik pada bagian akhir. Model terbaik serta hasil evaluasi disimpan ke `output_dir`.

**Penting:** label menit memakai asumsi interval 5 menit dari konfigurasi. File NPZ tanpa timestamp tidak dapat membuktikan interval ini. Satuan fisik flow mengikuti sumber dataset; jangan menganggapnya kendaraan/jam tanpa memeriksa metadata.

### Perbaikan dari versi awal

- Scaler, imputasi, dan graf dipelajari **hanya dari train**.
- Seluruh target setiap window berada di satu split; window terakhir ikut digunakan.
- Window dibuat saat dibaca sehingga tidak menggandakan seluruh dataset di memori.
- Operasi graf dan GRU diproses bersama untuk batch, waktu, dan sensor.
- Ada seed, gradient clipping, early stopping, scheduler, serta pemulihan checkpoint terbaik.
- Evaluasi mencakup MAE, RMSE, WAPE, error per horizon, dan pembanding persistence dalam skala asli.


## 01 · Lingkungan dan konfigurasi
Dependensi dipasang hanya jika belum tersedia. Implementasi GCN di bawah menggunakan PyTorch langsung sehingga `torch_geometric` tidak lagi wajib.


In [ ]:
import importlib.util
import subprocess
import sys

requirements = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "torch": "torch"}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import copy
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, HTML

@dataclass
class Config:
    data_path: str = "/content/drive/MyDrive/[CN] Colab Notebooks/[SML] Statistical ML/Spatio-Temporal GNN Traffic Flow Prediction/pems08.npz"
    output_dir: str = "outputs/pems08_stgnn"
    use_reference_data: bool = False  # True = data referensi, BUKAN hasil PEMS08.
    mount_drive: bool = True
    feature_names: tuple = ("Flow", "Occupancy", "Speed")
    target_feature: int = 0
    sample_minutes: int = 5
    input_steps: int = 12
    output_steps: int = 12
    train_ratio: float = 0.70
    val_ratio: float = 0.10
    corr_threshold: float = 0.60
    graph_top_k: int = 8
    gcn_hidden: int = 32
    gru_hidden: int = 64
    node_embedding_dim: int = 8
    dropout: float = 0.15
    batch_size: int = 32
    epochs: int = 40
    learning_rate: float = 0.001
    weight_decay: float = 0.0001
    patience: int = 8
    min_delta: float = 0.0001
    grad_clip: float = 1.0
    num_workers: int = 0  # Aman untuk notebook dan Colab.
    seed: int = 42
    plot_sensor: int = 0
    plot_points: int = 288
    save_predictions: bool = False  # Arsip prediksi lengkap dapat berukuran besar.

cfg = Config()
assert cfg.input_steps > 0 and cfg.output_steps > 0
assert 0 < cfg.train_ratio < 1 and 0 < cfg.val_ratio < 1 - cfg.train_ratio
assert cfg.sample_minutes > 0 and cfg.epochs > 0 and cfg.patience > 0
assert 0 <= cfg.corr_threshold <= 1 and cfg.graph_top_k >= 1
assert cfg.batch_size > 0 and 0 <= cfg.dropout < 1

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
# Seed membantu replikasi; kesamaan bit lintas GPU/versi library tidak dijamin.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_dir = Path(cfg.output_dir) / ("reference" if cfg.use_reference_data else "pems08")
run_dir.mkdir(parents=True, exist_ok=True)
RUN_LABEL = "REFERENCE REFERENSI · BUKAN PEMS08" if cfg.use_reference_data else "PEMS08"

COLORS = {"navy": "#173653", "teal": "#099e91", "orange": "#e5983b", "red": "#d25861", "muted": "#6e8191"}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 170, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.labelcolor": COLORS["navy"], "text.color": COLORS["navy"],
    "axes.grid": True, "grid.alpha": 0.18, "font.size": 10,
})
print(f"{RUN_LABEL} | device={device} | PyTorch {torch.__version__}")
print(f"Riwayat {cfg.input_steps * cfg.sample_minutes} menit → prediksi {cfg.output_steps * cfg.sample_minutes} menit")
print(f"Folder hasil: {run_dir.resolve()}")


## 02 · Muat dan periksa data
Format yang diharapkan: `data[time, sensor, feature]` dengan key NPZ `data` dan urutan fitur **flow, occupancy, speed** sesuai notebook awal. Nilai nol tetap dianggap pengamatan valid. NaN/inf pada input diimputasi; target yang tidak tersedia dikeluarkan dari loss dan metrik.


In [ ]:
def make_reference_data(seed=42, steps=960, nodes=12):
    """Pola buatan untuk smoke test; bukan pengganti dataset penelitian."""
    rng = np.random.default_rng(seed)
    t = np.arange(steps)[:, None]
    offsets = np.linspace(0, 0.7, nodes)[None, :]
    flow = np.maximum(0, 110 + 45 * np.sin(2 * np.pi * t / 288 + offsets) + rng.normal(0, 7, (steps, nodes)))
    occupancy = np.clip(0.04 + flow / 1400 + rng.normal(0, 0.007, flow.shape), 0, 1)
    speed = np.maximum(0, 75 - flow / 7 + rng.normal(0, 2, flow.shape))
    return np.stack([flow, occupancy, speed], axis=-1).astype(np.float32)

if cfg.use_reference_data:
    arr = make_reference_data(cfg.seed)
    warnings.warn("MODE REFERENCE: semua grafik dan metrik berasal dari data referensi.")
else:
    in_colab = "google.colab" in sys.modules
    if in_colab and cfg.mount_drive and cfg.data_path.startswith("/content/drive/"):
        from google.colab import drive
        drive.mount("/content/drive")
    data_path = Path(cfg.data_path).expanduser()
    if not data_path.is_file():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan: {data_path}\n"
            "Perbaiki cfg.data_path atau gunakan cfg.use_reference_data=True untuk reference referensi."
        )
    with np.load(data_path, allow_pickle=False) as data:
        if "data" not in data:
            raise KeyError(f"NPZ harus memiliki key 'data'. Key yang tersedia: {data.files}")
        arr = np.asarray(data["data"], dtype=np.float32)

if arr.ndim != 3 or arr.shape[2] != len(cfg.feature_names):
    raise ValueError(f"Diharapkan [time, sensor, {len(cfg.feature_names)} fitur], diterima {arr.shape}")
num_time, num_nodes, num_features = arr.shape
if num_nodes < 1 or not 0 <= cfg.target_feature < num_features:
    raise ValueError("Jumlah sensor atau indeks target tidak valid.")
if not 0 <= cfg.plot_sensor < num_nodes:
    raise ValueError(f"plot_sensor harus berada di 0..{num_nodes - 1}")

valid_mask = np.isfinite(arr)
quality_rows = []
for f, name in enumerate(cfg.feature_names):
    values = arr[:, :, f][valid_mask[:, :, f]]
    quality_rows.append({"Fitur": name, "Valid (%)": 100 * valid_mask[:, :, f].mean(),
                         "Minimum": values.min() if values.size else np.nan,
                         "Median": np.median(values) if values.size else np.nan,
                         "Maksimum": values.max() if values.size else np.nan})
print(f"{RUN_LABEL} | {num_time:,} timestep · {num_nodes} sensor · {num_features} fitur")
display(pd.DataFrame(quality_rows).round(3))


## 03 · Split kronologis dan preprocessing
Batas split ditentukan **sebelum** menghitung statistik. Median imputasi serta mean/std per fitur memakai train saja. Target NaN/inf tetap memiliki mask sehingga hasil imputasi tidak dianggap label asli.

Validation dan test boleh menggunakan riwayat input dari split sebelumnya, karena informasi itu sudah tersedia pada saat prediksi. Semua langkah **target** sebuah window harus tetap berada di split yang sama. Evaluasi adalah *rolling-origin*: setiap prediksi baru menerima pengamatan historis terbaru, bukan operasi satu forecast panjang tanpa pembaruan data.


In [ ]:
train_end = int(num_time * cfg.train_ratio)
val_end = train_end + int(num_time * cfg.val_ratio)
if train_end < cfg.input_steps + cfg.output_steps:
    raise ValueError("Split train terlalu pendek untuk panjang input dan horizon.")
if min(val_end - train_end, num_time - val_end) < cfg.output_steps:
    raise ValueError("Split validation/test harus memuat setidaknya satu horizon penuh.")

clean = np.where(valid_mask, arr, np.nan)
train_raw = clean[:train_end]
if np.any(np.isfinite(train_raw).sum(axis=(0, 1)) == 0):
    raise ValueError("Ada fitur tanpa nilai valid pada train; periksa dataset.")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    node_median = np.nanmedian(train_raw, axis=0)
    feature_median = np.nanmedian(train_raw, axis=(0, 1))
impute_values = np.where(np.isfinite(node_median), node_median, feature_median).astype(np.float32)
filled = np.where(valid_mask, arr, impute_values[None, :, :]).astype(np.float32)
train_mean = filled[:train_end].mean(axis=(0, 1), dtype=np.float64).astype(np.float32)
train_std = filled[:train_end].std(axis=(0, 1), dtype=np.float64).astype(np.float32)
train_std = np.where(train_std > 1e-6, train_std, 1.0).astype(np.float32)
scaled = np.ascontiguousarray((filled - train_mean) / train_std, dtype=np.float32)
target_mean = float(train_mean[cfg.target_feature])
target_std = float(train_std[cfg.target_feature])
assert np.isfinite(scaled).all()

split_ranges = {"Train": (0, train_end), "Validation": (train_end, val_end), "Test": (val_end, num_time)}
split_table = pd.DataFrame([
    {"Split": name, "Mulai inklusif": start, "Akhir eksklusif": end, "Timestep": end - start,
     "Target valid (%)": 100 * valid_mask[start:end, :, cfg.target_feature].mean()}
    for name, (start, end) in split_ranges.items()
])
display(split_table.round(2))


In [ ]:
# Grafik ini dibatasi ke train; tidak dipakai untuk memilih model berdasarkan test.
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), constrained_layout=True)
points = min(train_end, cfg.plot_points)
for f, ax in enumerate(axes):
    ax.plot(np.arange(points) * cfg.sample_minutes / 60,
            clean[:points, cfg.plot_sensor, f], color=COLORS["teal"], linewidth=1.4)
    ax.set(title=cfg.feature_names[f], xlabel="Jam sejak awal data", ylabel="Nilai asli")
fig.suptitle(f"{RUN_LABEL} | Profil train · sensor {cfg.plot_sensor}", fontsize=14, fontweight="bold")
fig.savefig(run_dir / "01_train_profile.png", bbox_inches="tight")
plt.show()


## 04 · Graf hubungan antarsensor
Graf diestimasi dari korelasi flow **pada train**. Setiap sensor memilih maksimal `graph_top_k` tetangga dengan korelasi positif di atas ambang, kemudian edge disimetriskan. Derajat setelah simetrisasi bisa melebihi top-k. Sensor tanpa tetangga tetap memiliki self-loop.

Graf korelasi menunjukkan kemiripan pola; ia **bukan** bukti keterhubungan jalan atau sebab-akibat. Jika tersedia adjacency fisik yang sesuai urutan sensor, gunakan itu dalam eksperimen terpisah.

Normalisasi GCN: $\hat A = D^{-1/2}(A+I)D^{-1/2}$. Untuk 170 sensor, matriks padat sederhana masih praktis. Untuk ribuan sensor, pertimbangkan operasi sparse; jumlah sensor besar meningkatkan biaya matriks secara kuadrat.


In [ ]:
def build_correlation_graph(train_flow, threshold=0.60, top_k=8):
    if train_flow.ndim != 2:
        raise ValueError("train_flow harus [time, sensor].")
    n = train_flow.shape[1]
    centered = train_flow.astype(np.float64) - train_flow.mean(axis=0)
    norms = np.linalg.norm(centered, axis=0)
    denom = np.outer(norms, norms)
    corr = np.divide(centered.T @ centered, denom, out=np.zeros((n, n)), where=denom > 1e-12)
    corr = np.clip(corr, -1, 1).astype(np.float32)
    np.fill_diagonal(corr, 1.0)
    weights = np.zeros_like(corr)
    for i in range(n):
        candidates = np.flatnonzero((corr[i] >= threshold) & (corr[i] > 0) & (np.arange(n) != i))
        chosen = candidates[np.argsort(-corr[i, candidates], kind="stable")[:top_k]]
        weights[i, chosen] = corr[i, chosen]
    weights = np.maximum(weights, weights.T)
    with_self = weights + np.eye(n, dtype=np.float32)
    degree = with_self.sum(axis=1)
    adj_norm = with_self / np.sqrt(degree[:, None] * degree[None, :])
    return corr, weights, adj_norm.astype(np.float32)

corr_matrix, graph_weights, adj_norm = build_correlation_graph(
    filled[:train_end, :, cfg.target_feature], cfg.corr_threshold, cfg.graph_top_k
)
neighbors = (graph_weights > 0).sum(axis=1)
edges = int(np.count_nonzero(np.triu(graph_weights, 1)))
possible_edges = num_nodes * (num_nodes - 1) // 2
print(f"{num_nodes} sensor | {edges:,} edge tak berarah | kepadatan {edges / max(possible_edges, 1):.1%}")
print(f"Tetangga min/median/max: {neighbors.min()} / {np.median(neighbors):.0f} / {neighbors.max()}")
print(f"Sensor tanpa tetangga (tetap memakai self-loop): {int((neighbors == 0).sum())}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
im = axes[0].imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set(title="Korelasi pada train", xlabel="Sensor", ylabel="Sensor")
fig.colorbar(im, ax=axes[0], shrink=0.8, label="Pearson r")
im = axes[1].imshow(graph_weights, cmap="Blues", vmin=0, vmax=1)
axes[1].set(title="Graf terpilih · tanpa self-loop", xlabel="Sensor", ylabel="Sensor")
fig.colorbar(im, ax=axes[1], shrink=0.8, label="Bobot edge")
for ax in axes:
    ax.grid(False)
fig.suptitle(RUN_LABEL, fontsize=13, fontweight="bold")
fig.savefig(run_dir / "02_graph.png", bbox_inches="tight")
plt.show()


## 05 · Window dan DataLoader
Bentuk batch: input `[B, L, N, F]`, target/mask `[B, H, N]`. `L` adalah panjang riwayat, `H` horizon, dan `N` jumlah sensor. Pengacakan hanya diterapkan pada window train; urutan waktu di dalam window tidak berubah.


In [ ]:
class TrafficWindowDataset(Dataset):
    def __init__(self, values, observed, start, end, input_steps, output_steps, target_feature):
        self.values = values
        self.observed = observed
        self.input_steps = input_steps
        self.output_steps = output_steps
        self.target_feature = target_feature
        first_origin = max(start, input_steps)
        last_origin = end - output_steps
        self.origins = np.arange(first_origin, last_origin + 1, dtype=np.int64)
        if not len(self.origins):
            raise ValueError(f"Tidak ada window valid untuk rentang [{start}, {end}).")

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, index):
        t = int(self.origins[index])
        x = self.values[t - self.input_steps:t]
        y = self.values[t:t + self.output_steps, :, self.target_feature]
        mask = self.observed[t:t + self.output_steps, :, self.target_feature]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(mask)

datasets = {
    name: TrafficWindowDataset(scaled, valid_mask, start, end, cfg.input_steps, cfg.output_steps, cfg.target_feature)
    for name, (start, end) in split_ranges.items()
}
loader_options = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=device.type == "cuda")
train_loader = DataLoader(datasets["Train"], shuffle=True,
                          generator=torch.Generator().manual_seed(cfg.seed), **loader_options)
val_loader = DataLoader(datasets["Validation"], shuffle=False, **loader_options)
test_loader = DataLoader(datasets["Test"], shuffle=False, **loader_options)
for name, ds in datasets.items():
    start, end = split_ranges[name]
    assert ds.origins[0] >= start and ds.origins[-1] + cfg.output_steps <= end
    assert ds.origins[-1] + cfg.output_steps == end
    print(f"{name:12s}: {len(ds):,} window | awal target {ds.origins[0]}..{ds.origins[-1]}")


## 06 · Model GCN + GRU

| Tahap | Fungsi | Bentuk keluaran |
|---|---|---|
| Proyeksi input | Ubah fitur menjadi embedding | B × L × N × C |
| Dua GCN residual | Gabungkan informasi tetangga tanpa membuang sinyal lokal | B × L × N × C |
| Fusi | Gabungkan hasil GCN, fitur asli, dan embedding sensor | B × L × N × (C + F + E) |
| GRU bersama | Pelajari urutan waktu setiap sensor | (B × N) × R |
| Head residual | Prediksi koreksi terhadap flow terakhir | B × H × N |

Bobot GRU dipakai bersama antarsensor. Head awal bernilai nol sehingga prediksi awal sama dengan persistence. Seluruh horizon diprediksi sekaligus; model tidak menggunakan target masa depan sebagai input. Embedding sensor mengharuskan jumlah dan **urutan sensor tetap sama** saat inference.


In [ ]:
class SpatioTemporalGNN(nn.Module):
    def __init__(self, in_features, n_nodes, adjacency, config):
        super().__init__()
        self.n_nodes = n_nodes
        self.horizon = config.output_steps
        self.target_feature = config.target_feature
        self.register_buffer("adj_norm", torch.as_tensor(adjacency, dtype=torch.float32))
        self.input_proj = nn.Linear(in_features, config.gcn_hidden)
        self.graph_layers = nn.ModuleList([nn.Linear(config.gcn_hidden, config.gcn_hidden) for _ in range(2)])
        self.norms = nn.ModuleList([nn.LayerNorm(config.gcn_hidden) for _ in range(2)])
        self.node_embedding = nn.Parameter(torch.empty(n_nodes, config.node_embedding_dim))
        nn.init.normal_(self.node_embedding, std=0.02)
        self.dropout = nn.Dropout(config.dropout)
        self.gru = nn.GRU(config.gcn_hidden + in_features + config.node_embedding_dim,
                          config.gru_hidden, batch_first=True)
        self.head = nn.Linear(config.gru_hidden, config.output_steps)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        b, length, n, _ = x.shape
        if n != self.n_nodes:
            raise ValueError("Jumlah/urutan sensor harus sama dengan training.")
        h = torch.relu(self.input_proj(x))
        for linear, norm in zip(self.graph_layers, self.norms):
            # A[N,N] @ H[B,L,N,C]: seluruh batch dan timestep diproses sekaligus.
            spatial = torch.matmul(self.adj_norm, linear(h))
            h = norm(h + self.dropout(torch.relu(spatial)))
        node_ids = self.node_embedding[None, None].expand(b, length, -1, -1)
        fused = torch.cat([h, x, node_ids], dim=-1)
        sequence = fused.permute(0, 2, 1, 3).reshape(b * n, length, -1)
        _, hidden = self.gru(sequence)
        correction = self.head(self.dropout(hidden[-1])).reshape(b, n, self.horizon).permute(0, 2, 1)
        last_flow = x[:, -1, :, self.target_feature].unsqueeze(1)
        return last_flow + correction

model = SpatioTemporalGNN(num_features, num_nodes, adj_norm, cfg).to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
x_sample, _, _ = next(iter(val_loader))
model.eval()
with torch.inference_mode():
    sample_prediction = model(x_sample.to(device))
assert sample_prediction.shape == (len(x_sample), cfg.output_steps, num_nodes)
print(f"Parameter trainable: {parameter_count:,}")
print(f"Input: {tuple(x_sample.shape)} → output: {tuple(sample_prediction.shape)}")


## 07 · Training dan pemilihan checkpoint
Loss memakai MSE yang dinormalisasi dan hanya menghitung target valid. Loss epoch dibobot berdasarkan jumlah target valid, sehingga batch terakhir tidak mendapat bobot berlebih. Model dipilih berdasarkan **validation loss**, bukan test. `epochs` adalah batas maksimum; early stopping dapat menghentikan lebih awal.


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    squared_error, valid_count = 0.0, 0
    with torch.set_grad_enabled(training):
        for x, y, mask in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            count = int(mask.sum().item())
            if count == 0:
                continue
            if training:
                optimizer.zero_grad(set_to_none=True)
            prediction = model(x)
            loss = (prediction[mask] - y[mask]).square().mean()
            if not torch.isfinite(loss):
                raise FloatingPointError("Loss tidak finite; periksa input dan learning rate.")
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip, error_if_nonfinite=True)
                optimizer.step()
            squared_error += loss.item() * count
            valid_count += count
    if valid_count == 0:
        raise ValueError("Split tidak memiliki target valid.")
    return squared_error / valid_count

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
best_val = float("inf")
best_state = None
best_epoch = 0
patience_reference = float("inf")
wait = 0
history_rows = []
started = time.perf_counter()

for epoch in range(1, cfg.epochs + 1):
    tick = time.perf_counter()
    learning_rate_used = optimizer.param_groups[0]["lr"]
    train_loss = run_epoch(model, train_loader, optimizer)
    val_loss = run_epoch(model, val_loader)
    scheduler.step(val_loss)
    history_rows.append({"epoch": epoch, "train_mse": train_loss, "val_mse": val_loss,
                         "learning_rate": learning_rate_used, "seconds": time.perf_counter() - tick})
    # Simpan minimum absolut, terpisah dari ambang kesabaran early stopping.
    if val_loss < best_val:
        best_val, best_epoch = val_loss, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if val_loss < patience_reference - cfg.min_delta:
        patience_reference, wait = val_loss, 0
    else:
        wait += 1
    print(f"Epoch {epoch:02d}/{cfg.epochs} | train {train_loss:.5f} | val {val_loss:.5f} | "
          f"lr {learning_rate_used:.2g} | {history_rows[-1]['seconds']:.1f}s")
    if wait >= cfg.patience:
        print(f"Early stopping; {cfg.patience} epoch tanpa perbaikan validation yang cukup.")
        break

model.load_state_dict(best_state)
model.eval()
history = pd.DataFrame(history_rows)
print(f"Checkpoint terbaik: epoch {best_epoch}, validation MSE {best_val:.5f}")
print(f"Waktu training: {(time.perf_counter() - started) / 60:.2f} menit")


## 08 · Evaluasi pada test yang belum digunakan untuk pemilihan model
Prediksi dikembalikan ke **skala asli**. Pembanding persistence memakai nilai target terakhir, atau hasil imputasi train jika input terakhir hilang. Tidak ada clipping nilai negatif pada metrik, sehingga evaluasi mengukur output model apa adanya.

- **MAE:** rata-rata besar kesalahan; lebih kecil lebih baik.
- **RMSE:** memberi penalti lebih besar pada kesalahan besar.
- **WAPE:** $100 \times \sum|y-\hat y|/\sum|y|$. Menghindari pembagian per titik bernilai nol seperti pada MAPE; menjadi NaN jika seluruh aktual nol.
- **MAE skill (%):** peningkatan terhadap persistence; positif berarti model lebih baik, negatif berarti lebih buruk.

Metrik agregat menghitung semua pasangan origin–horizon–sensor valid. Karena window overlap, timestep yang sama dapat muncul sebagai target pada horizon berbeda. Tabel per horizon membantu membaca perbedaannya.


In [ ]:
@torch.inference_mode()
def collect_predictions(model, loader):
    predictions, targets, masks, baselines = [], [], [], []
    model.eval()
    for x, y, mask in loader:
        prediction = model(x.to(device)).cpu().numpy()
        persistence = x[:, -1, :, cfg.target_feature].numpy()[:, None, :]
        predictions.append(prediction)
        targets.append(y.numpy())
        masks.append(mask.numpy())
        baselines.append(np.repeat(persistence, cfg.output_steps, axis=1))
    inverse = lambda z: z * target_std + target_mean
    return (inverse(np.concatenate(predictions)), inverse(np.concatenate(targets)),
            np.concatenate(masks), inverse(np.concatenate(baselines)))

def regression_metrics(y_true, y_pred, mask):
    observed = np.asarray(mask, dtype=bool) & np.isfinite(y_true)
    if not observed.any():
        return {"MAE": np.nan, "RMSE": np.nan, "WAPE (%)": np.nan, "N valid": 0}
    if not np.isfinite(y_pred[observed]).all():
        raise FloatingPointError("Prediksi memiliki NaN/inf pada target valid.")
    actual = np.asarray(y_true[observed], dtype=np.float64)
    error = np.asarray(y_pred[observed], dtype=np.float64) - actual
    denominator = np.abs(actual).sum()
    return {"MAE": float(np.abs(error).mean()), "RMSE": float(np.sqrt(np.square(error).mean())),
            "WAPE (%)": float(100 * np.abs(error).sum() / denominator) if denominator > 1e-12 else np.nan,
            "N valid": int(observed.sum())}

pred_test, true_test, mask_test, baseline_test = collect_predictions(model, test_loader)
summary = pd.DataFrame([
    {"Model": "Persistence", **regression_metrics(true_test, baseline_test, mask_test)},
    {"Model": "STGNN", **regression_metrics(true_test, pred_test, mask_test)},
]).set_index("Model")
base_mae = summary.loc["Persistence", "MAE"]
summary["MAE skill (%)"] = 100 * (1 - summary["MAE"] / base_mae) if base_mae > 1e-12 else np.nan
print(f"TEST · {RUN_LABEL} | checkpoint epoch {best_epoch}")
display(summary.round(3))

horizon_rows = []
for h in range(cfg.output_steps):
    for name, prediction in [("Persistence", baseline_test), ("STGNN", pred_test)]:
        horizon_rows.append({"Model": name, "Langkah": h + 1, "Menit": (h + 1) * cfg.sample_minutes,
                             **regression_metrics(true_test[:, h], prediction[:, h], mask_test[:, h])})
horizon_metrics = pd.DataFrame(horizon_rows)
display(horizon_metrics.round(3))
if summary.loc["STGNN", "MAE"] >= base_mae:
    print("Pada test ini STGNN belum mengungguli persistence. Jangan menyimpulkan perbaikan akurasi.")


## 09 · Dashboard hasil
Grafik aktual/prediksi memakai horizon pertama dan sensor pilihan. Error per sensor memakai seluruh horizon. Grafik merupakan diagnosis setelah evaluasi; keputusan tuning berikutnya harus memakai validation, kemudian dievaluasi pada periode test baru jika test lama sudah sering diperiksa.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
ax = axes[0, 0]
ax.plot(history["epoch"], history["train_mse"], label="Train", color=COLORS["teal"], linewidth=2)
ax.plot(history["epoch"], history["val_mse"], label="Validation", color=COLORS["orange"], linewidth=2)
ax.axvline(best_epoch, color=COLORS["muted"], linestyle=":", label=f"Terbaik: {best_epoch}")
ax.set(title="01  Proses belajar", xlabel="Epoch", ylabel="Masked MSE · skala normalisasi")
ax.legend(frameon=False)

ax = axes[0, 1]
for name, color in [("Persistence", COLORS["orange"]), ("STGNN", COLORS["teal"])]:
    rows = horizon_metrics[horizon_metrics["Model"] == name]
    ax.plot(rows["Menit"], rows["MAE"], marker="o", color=color, label=name, linewidth=2)
ax.set(title="02  Error menurut horizon", xlabel="Menit ke depan (sesuai konfigurasi)", ylabel="MAE · skala asli")
ax.legend(frameon=False)

ax = axes[1, 0]
n_plot = min(cfg.plot_points, len(true_test))
sensor = cfg.plot_sensor
hours = np.arange(n_plot) * cfg.sample_minutes / 60
observed = np.where(mask_test[:n_plot, 0, sensor], true_test[:n_plot, 0, sensor], np.nan)
ax.plot(hours, observed, color=COLORS["navy"], label="Aktual", linewidth=1.5)
ax.plot(hours, pred_test[:n_plot, 0, sensor], color=COLORS["teal"], label="STGNN", alpha=0.9)
ax.plot(hours, baseline_test[:n_plot, 0, sensor], color=COLORS["orange"], label="Persistence", alpha=0.55, linestyle="--")
ax.set(title=f"03  Sensor {sensor} · horizon {cfg.sample_minutes} menit", xlabel="Jam sejak target test pertama", ylabel="Flow · skala asli")
ax.legend(frameon=False, ncol=3, fontsize=9)

ax = axes[1, 1]
abs_error = np.abs(pred_test - true_test)
counts = mask_test.sum(axis=(0, 1))
mae_by_sensor = np.divide(np.where(mask_test, abs_error, 0).sum(axis=(0, 1)), counts,
                          out=np.full(num_nodes, np.nan), where=counts > 0)
valid_sensors = np.flatnonzero(np.isfinite(mae_by_sensor))
worst = valid_sensors[np.argsort(mae_by_sensor[valid_sensors])[-min(10, len(valid_sensors)):]]
ax.barh([str(i) for i in worst], mae_by_sensor[worst], color=COLORS["teal"], alpha=0.85)
ax.set(title="04  Sensor dengan error terbesar", xlabel="MAE · seluruh horizon", ylabel="Indeks sensor")
fig.suptitle(f"{RUN_LABEL} | Evaluasi STGNN", fontsize=18, fontweight="bold")
fig.savefig(run_dir / "03_evaluation_dashboard.png", bbox_inches="tight")
plt.show()


## 10 · Simpan eksperimen
Checkpoint mencakup bobot, graf, statistik preprocessing, konfigurasi, dan batas split. Folder reference dipisahkan dari hasil PEMS08. Nama file dalam folder run yang sama akan diperbarui saat sel dijalankan ulang; ubah `output_dir` untuk menyimpan eksperimen terpisah. Checkpoint ini untuk inference, bukan resume optimizer persis dari titik terakhir.


In [ ]:
checkpoint = {
    "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
    "config": asdict(cfg), "num_nodes": num_nodes, "num_features": num_features,
    "feature_names": list(cfg.feature_names), "target_mean": target_mean, "target_std": target_std,
    "train_mean": torch.from_numpy(train_mean.copy()), "train_std": torch.from_numpy(train_std.copy()),
    "impute_values": torch.from_numpy(impute_values.copy()),
    "adj_norm": torch.from_numpy(adj_norm.copy()), "graph_weights": torch.from_numpy(graph_weights.copy()),
    "train_end": train_end, "val_end": val_end, "best_epoch": best_epoch, "best_val_mse": best_val,
    "run_label": RUN_LABEL, "torch_version": str(torch.__version__),
}
checkpoint_path = run_dir / "best_model.pt"
torch.save(checkpoint, checkpoint_path)
history.to_csv(run_dir / "training_history.csv", index=False)
summary.to_csv(run_dir / "test_metrics.csv")
horizon_metrics.to_csv(run_dir / "horizon_metrics.csv", index=False)
pd.DataFrame({"sensor": np.arange(num_nodes), "MAE": mae_by_sensor, "N_valid": counts}).to_csv(run_dir / "sensor_metrics.csv", index=False)
with (run_dir / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(cfg), handle, indent=2, ensure_ascii=False)
if cfg.save_predictions:
    np.savez_compressed(run_dir / "test_predictions.npz", predictions=pred_test, actual=true_test,
                        observed=mask_test, persistence=baseline_test, origins=datasets["Test"].origins)
print(f"Tersimpan: {checkpoint_path.resolve()}")
print("CSV metrik, konfigurasi, serta tiga grafik PNG tersedia pada folder run yang sama.")


## 11 · Inference dari checkpoint
Masukan harus memiliki urutan fitur dan sensor yang sama dengan training. Untuk runtime baru, jalankan definisi `Config` dan `SpatioTemporalGNN`, kemudian fungsi berikut; pelatihan tidak perlu diulang. Forecast dari pengamatan paling akhir belum memiliki ground truth masa depan sehingga tidak disertai metrik akurasi.


In [ ]:
def load_forecaster(checkpoint_path, inference_device="cpu"):
    # Gunakan checkpoint eksperimen sendiri atau sumber yang dipercaya.
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    loaded_config = Config(**saved["config"])
    loaded_model = SpatioTemporalGNN(saved["num_features"], saved["num_nodes"], saved["adj_norm"], loaded_config)
    loaded_model.load_state_dict(saved["model_state"])
    loaded_model.to(inference_device).eval()
    return loaded_model, saved

@torch.inference_mode()
def forecast_next(raw_history, trained_model, saved):
    config = saved["config"]
    expected = (config["input_steps"], saved["num_nodes"], saved["num_features"])
    raw_history = np.asarray(raw_history, dtype=np.float32)
    if raw_history.shape != expected:
        raise ValueError(f"Riwayat harus berbentuk {expected}, diterima {raw_history.shape}")
    medians = saved["impute_values"].cpu().numpy()
    means = saved["train_mean"].cpu().numpy()
    stds = saved["train_std"].cpu().numpy()
    values = np.where(np.isfinite(raw_history), raw_history, medians[None])
    values = np.ascontiguousarray((values - means) / stds, dtype=np.float32)
    model_device = next(trained_model.parameters()).device
    trained_model.eval()
    prediction = trained_model(torch.from_numpy(values).unsqueeze(0).to(model_device))[0].cpu().numpy()
    return prediction * saved["target_std"] + saved["target_mean"]

restored_model, saved = load_forecaster(checkpoint_path, str(device))
future_flow = forecast_next(arr[-cfg.input_steps:], restored_model, saved)
forecast_table = pd.DataFrame(future_flow, columns=[f"sensor_{i}" for i in range(num_nodes)])
forecast_table.index = pd.Index(np.arange(1, cfg.output_steps + 1) * cfg.sample_minutes, name="menit_ke_depan")
forecast_table.to_csv(run_dir / "next_forecast.csv")
print(f"{RUN_LABEL} | forecast terbaru, {num_nodes} sensor; preview maksimal 6 sensor")
display(forecast_table.iloc[:, :min(6, num_nodes)].round(2))


## Membaca hasil dan eksperimen berikutnya

- **MAE skill positif:** model mengungguli persistence untuk test ini. Tetap lihat error per horizon dan sensor.
- **Train membaik, validation memburuk:** indikasi overfitting; gunakan checkpoint terbaik dan uji kapasitas/dropout pada validation.
- **Model kalah dari persistence:** hasil ini tetap informatif. Uji horizon, panjang riwayat, graf fisik, atau model temporal tanpa graf dengan split yang sama.
- **Prediksi flow negatif:** model regresi tidak memiliki batas nonnegatif. Metrik di notebook ini memakai output mentah. Bila menerapkan clipping pada aplikasi, laporkan metrik clipping secara terpisah.

Untuk laporan, catat konfigurasi, seed, batas split, cara menangani data hilang, dan versi library. Jangan menyamakan traffic flow tinggi dengan kemacetan secara langsung: penentuan kemacetan memerlukan interpretasi speed/occupancy dan kondisi jalan.

**Batas implementasi:** normalisasi dan urutan sensor bersifat tetap; belum ada penanganan sensor baru, drift, atau layanan data real-time. Mode reference hanya membuktikan alur kode dapat berjalan, bukan kualitas prediksi dunia nyata.
